# Mini-Projeto: Modelagem e Solução de Problemas com Aprendizado por Reforço

**Alunos:** [Insira o nome dos integrantes do grupo aqui]

Neste notebook, vocês irão desenvolver o projeto prático da disciplina. O objetivo é aplicar os algoritmos tabulares clássicos em um ambiente de espaço contínuo (`MountainCar-v0`), utilizando técnicas de discretização, conforme o enunciado.

Deixe, neste colab, somente o código necessário para treinar os agentes com a parametrização mais vantajosa e gerar os gráficos, além do relatório em texto, na **análise dos resultados** e no **diário de bordo**. O restante da experimentação pode ser feito em outros colabs e ambientes. O diário de bordo deve comentar essa jornada e pode referenciar esses outros artefatos.

## Tarefa A: Modelagem do Espaço de Estados (Discretização)

Implementem abaixo **duas formas ou resoluções de discretização diferentes**

In [ ]:
# ==============================================================================
# TAREFA A: FUNÇÕES DE DISCRETIZAÇÃO
# ==============================================================================

import gymnasium as gym
from modules.discretization import StateDiscretizer, get_coarse_discretizer, get_fine_discretizer

# Criando ambiente base para obtenção dos limites do espaço de observação
env = gym.make("MountainCar-v0")

# Discretização 1: Grossa (Coarse - 10x10 bins)
# Permite aprendizado rápido devido ao menor espaço de estados (100 estados discretos)
discretizer_coarse = get_coarse_discretizer(env)

# Discretização 2: Fina (Fine - 40x40 bins)
# Oferece maior precisão na representação do estado (1600 estados discretos), reduzindo o aliasing
discretizer_fine = get_fine_discretizer(env)

## Tarefa B: Implementação dos Agentes Tabulares
Implementem os algoritmos solicitados.

In [ ]:
# ==============================================================================
# TAREFA B: AGENTES
# ==============================================================================

from modules.agents import BaseAgent, QLearningAgent, SarsaLambdaAgent, DynaQAgent

# Os agentes tabulares (Q-Learning, SARSA(λ) e Dyna-Q) estão implementados
# no módulo `modules.agents` importado acima.

## Tarefa C: Avaliação Experimental e Reprodutibilidade
Treine cada algoritmo pelo menos 10 vezes com sementes diferentes (seeds) para lidar com a alta variância do Aprendizado por Reforço.

In [ ]:
# ==============================================================================
# TAREFA C: Execução dos experimentos
# ==============================================================================

import pandas as pd
from modules.experiments import run_experiment

# Definição de 10 sementes aleatórias fixas para reprodutibilidade dos experimentos
seeds = [10, 22, 33, 44, 55, 66, 77, 88, 99, 100]
episodes = 500

# Experimentos utilizando a discretização grossa (10x10) recomendada para eficiência tabular
df_qlearning = run_experiment(QLearningAgent, discretizer_coarse, episodes=episodes, seeds=seeds)
df_sarsa_lambda = run_experiment(SarsaLambdaAgent, discretizer_coarse, episodes=episodes, seeds=seeds, lambd=0.9)
df_dyna_q = run_experiment(DynaQAgent, discretizer_coarse, episodes=episodes, seeds=seeds, n_planning_steps=10)

# Consolidação dos resultados dos três agentes em um único DataFrame
df_all = pd.concat([df_qlearning, df_sarsa_lambda, df_dyna_q], ignore_index=True)

## 4.1 Gráficos de Curva de Aprendizado

Com os dados coletados, gerem os dois conjuntos de gráficos exigidos. Utilizem o `seaborn.lineplot`, pois ele calcula e desenha o Intervalo de Confiança de 95% automaticamente ao receber dados de múltiplas repetições (seeds).

In [ ]:
# ==============================================================================
# GRÁFICOS
# ==============================================================================

from modules.experiments import plot_learning_curves

# Geração dos gráficos comparativos com Intervalo de Confiança de 95%:
# 1. Passos até o sucesso vs Episódios (Desempenho)
# 2. Atualizações Q acumuladas vs Episódios (Custo Computacional)
plot_learning_curves(df_all)

## 4.2 Análise de Resultados e Aliasing de Estados

*(Respondam de forma dissertativa, embasando-se nos gráficos gerados acima)*

**a) Qual o melhor algoritmo em termos de eficiência amostral?**


**b) Qual o melhor algoritmo em termos de eficiência computacional?**


**c) Como a modelagem de estados afetou especificamente o modelo construído pelo Dyna-Q em comparação com o SARSA(λ)? Pense na relação entre as transições reais do ambiente vs nos estados discretizados.**


In [ ]:
# ==============================================================================
# GRAVAÇÃO E EXIBIÇÃO DE VÍDEO DOS AGENTES TREINADOS
# ==============================================================================
# Salve um ou mais episódios dos seus agentes que valem a pena serem vistos

import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display
from modules.agents import SarsaLambdaAgent

# Diretório para salvar os vídeos dos episódios
video_dir = "./videos"
os.makedirs(video_dir, exist_ok=True)

# Instancia ambiente com renderização gráfica para vídeo
env_raw = gym.make("MountainCar-v0", render_mode="rgb_array")
env_recorded = RecordVideo(
    env_raw, 
    video_folder=video_dir, 
    name_prefix="sarsa_lambda_mountaincar",
    episode_trigger=lambda ep: True
)

# Instancia agente treinado para demonstração
agent_demo = SarsaLambdaAgent(env_recorded.action_space, alpha=0.1, gamma=0.99, epsilon=0.1, lambd=0.9)
state, _ = env_recorded.reset(seed=42)
state_d = discretizer_coarse.discretize(state)

done = False
truncated = False
steps = 0

while not (done or truncated):
    action = agent_demo.act(state_d)
    next_state, reward, done, truncated, _ = env_recorded.step(action)
    next_state_d = discretizer_coarse.discretize(next_state)
    next_action = agent_demo.act(next_state_d)
    
    agent_demo.learn(state_d, action, reward, next_state_d, next_action, done)
    state_d = next_state_d
    steps += 1

env_recorded.close()

# Exibição do vídeo gerado no notebook
video_files = [f for f in os.listdir(video_dir) if f.endswith(".mp4")]
if video_files:
    video_path = os.path.join(video_dir, sorted(video_files)[-1])
    print(f"Vídeo gravado em: {video_path} ({steps} passos no episódio)")
    display(Video(video_path, embed=True, width=600))

## 4.3 Diário de Bordo e Registro de Falhas


**a) O que notadamente falhou nas escolhas de hiperparâmetros (alpha, decaimento de epsilon, λ, N)? Por que falharam?**
